# Lecture 4 — Functions

Today’s goal:

> Stop writing all logic inside one place and learn how to make reusable metric calculators.

This is a big step. After this lecture, your code starts to look much closer to real Autoware metric code.

## 1. Why do we need functions?

So far, we wrote everything in one block.

That is okay for tiny practice. But real code would become messy:

```cpp
// speed metric logic
// TTC metric logic
// lane keeping metric logic
// collision metric logic
// comfort metric logic
// progress metric logic
```

Disaster spaghetti. C++ pasta shop.

Instead, we separate logic into functions.

## 2. Basic function shape

A function has this shape:

```cpp
return_type function_name(input_arguments)
{
  // logic
  return value;
}
```

Example:

```cpp
double add(double a, double b)
{
  return a + b;
}
```

In [1]:
#include <iostream>

// In notebooks, lambdas are easy to re-run without redefinition problems.
{
  auto add = [](double a, double b) -> double {
    return a + b;
  };

  double result = add(3.0, 4.0);
  std::cout << "result = " << result << std::endl;
}

result = 7


The real C++ function syntax is:

```cpp
double add(double a, double b)
{
  return a + b;
}
```

Meaning:

```text
return_type: double
function_name: add
inputs: double a, double b
output: a + b
```

Notebook note: if you define the same function twice in a C++ kernel, you may get a redefinition error. That is why some runnable cells use lambdas inside `{}` blocks.

## 3. Function must be known before use

In a `.cpp` file, this works:

```cpp
double add(double a, double b)
{
  return a + b;
}

int main()
{
  double result = add(3.0, 4.0);
}
```

But if `main()` appears before C++ knows `add()` exists, it may fail.

Later, we solve this using **function declarations** and `.hpp` files. For now, helper functions are conceptually written above the place where they are used.

## 4. Metric function: speed score

Rule:

```text
If speed > 10:
    score = 0
Otherwise:
    score = 1
```

In [2]:
#include <iostream>

{
  auto calculate_speed_score = [](double ego_speed_mps) -> double {
    const double speed_limit_mps = 10.0;

    if (ego_speed_mps > speed_limit_mps) {
      return 0.0;
    }

    return 1.0;
  };

  double ego_speed_mps = 12.0;
  double score = calculate_speed_score(ego_speed_mps);

  std::cout << "score = " << score << std::endl;
}

score = 0


## 5. Important idea: `return` exits immediately

If speed is too high:

```cpp
return 0.0;
```

The function immediately exits.

It never reaches:

```cpp
return 1.0;
```

This is called **early return**. This pattern is extremely common in metrics.

## 6. Function with multiple inputs

Sometimes one input is not enough.

Instead of hardcoding the threshold, pass it as an argument.

In [3]:
#include <iostream>

{
  auto calculate_speed_score = [](double ego_speed_mps, double speed_limit_mps) -> double {
    if (ego_speed_mps > speed_limit_mps) {
      return 0.0;
    }

    return 1.0;
  };

  double ego_speed_mps = 12.0;
  double speed_limit_mps = 10.0;

  double score = calculate_speed_score(ego_speed_mps, speed_limit_mps);

  std::cout << "score = " << score << std::endl;
}

score = 0


## 7. Function returning `bool`

Functions can return `bool`.

Function names that return `bool` often start with:

```text
is_
has_
can_
should_
```

Examples:

```cpp
bool is_speed_too_high(...);
bool has_lane_info(...);
bool is_collision_detected(...);
bool is_trajectory_empty(...);
```

In [4]:
#include <iostream>

{
  auto is_speed_too_high = [](double ego_speed_mps, double speed_limit_mps) -> bool {
    return ego_speed_mps > speed_limit_mps;
  };

  double ego_speed_mps = 12.0;
  double speed_limit_mps = 10.0;

  bool too_high = is_speed_too_high(ego_speed_mps, speed_limit_mps);

  std::cout << std::boolalpha;
  std::cout << "too_high = " << too_high << std::endl;
}

too_high = true


## 8. Function returning `std::string`

A function can return text.

In [5]:
#include <iostream>
#include <string>

{
  auto get_speed_reason = [](double ego_speed_mps, double speed_limit_mps) -> std::string {
    if (ego_speed_mps > speed_limit_mps) {
      return "speed_too_high";
    }

    return "available";
  };

  double ego_speed_mps = 12.0;
  double speed_limit_mps = 10.0;

  std::string reason = get_speed_reason(ego_speed_mps, speed_limit_mps);

  std::cout << "reason = " << reason << std::endl;
}

reason = speed_too_high


## 9. Problem: score, available, reason are separated

So far, we can return one thing:

```cpp
double
bool
std::string
```

But a metric usually needs multiple outputs:

```text
score
available
reason
```

With separate functions, we might do this:

```cpp
double score = calculate_speed_score(speed, limit);
std::string reason = get_speed_reason(speed, limit);
bool available = true;
```

That works, but it can become inconsistent.

Example bug:

```cpp
double score = 0.0;
std::string reason = "available";
```

Oops. Score says fail, reason says available.

This is why real metric code usually returns a **result object**. Next lecture fixes this with `struct`.

## 10. `void` function: returns nothing

Sometimes a function does not return a value. Use:

```cpp
void
```

Printing is a good example.

In [6]:
#include <iostream>
#include <string>

{
  auto print_metric_result = [](double score, bool available, std::string reason) -> void {
    std::cout << std::boolalpha;
    std::cout << "score = " << score << std::endl;
    std::cout << "available = " << available << std::endl;
    std::cout << "reason = " << reason << std::endl;
  };

  double score = 1.0;
  bool available = true;
  std::string reason = "available";

  print_metric_result(score, available, reason);
}

score = 1
available = true
reason = available


## 11. Full metric-like example with functions

This combines bool, score, reason, and printing.

In [7]:
#include <iostream>
#include <string>

{
  auto is_ttc_too_low = [](double ttc_s, double threshold_s) -> bool {
    return ttc_s < threshold_s;
  };

  auto calculate_ttc_score = [&](double ttc_s, double threshold_s) -> double {
    if (is_ttc_too_low(ttc_s, threshold_s)) {
      return 0.0;
    }

    return 1.0;
  };

  auto get_ttc_reason = [&](double ttc_s, double threshold_s) -> std::string {
    if (is_ttc_too_low(ttc_s, threshold_s)) {
      return "ttc_too_low";
    }

    return "available";
  };

  const double threshold_s = 2.0;
  double ttc_s = 1.5;

  bool available = true;
  double score = calculate_ttc_score(ttc_s, threshold_s);
  std::string reason = get_ttc_reason(ttc_s, threshold_s);

  std::cout << std::boolalpha;
  std::cout << "score = " << score << std::endl;
  std::cout << "available = " << available << std::endl;
  std::cout << "reason = " << reason << std::endl;
}

score = 0
available = true
reason = ttc_too_low


## 12. Function naming

Bad:

```cpp
double f(double a, double b)
```

Better:

```cpp
double calculate_ttc_score(double ttc_s, double threshold_s)
```

Good names reduce brain load.

In metric code, prefer names like:

```cpp
calculate_ego_progress(...)
calculate_lane_keeping(...)
calculate_ttc_within_bound(...)
is_collision_detected(...)
is_trajectory_empty(...)
compute_lateral_deviation_m(...)
```

## 13. Common beginner mistakes

### Mistake 1: forgetting return value

Wrong:

```cpp
double calculate_score(double speed_mps)
{
  if (speed_mps > 10.0) {
    return 0.0;
  }
}
```

Problem: what if `speed_mps <= 10.0`?

Correct:

```cpp
double calculate_score(double speed_mps)
{
  if (speed_mps > 10.0) {
    return 0.0;
  }

  return 1.0;
}
```

### Mistake 2: return type mismatch

Wrong:

```cpp
double get_reason()
{
  return "available";
}
```

Correct:

```cpp
std::string get_reason()
{
  return "available";
}
```

## 14. Homework

Write a simple function-based TTC metric.

Rules:

```text
Function 1:
is_ttc_too_low(ttc_s, threshold_s)
returns true if ttc_s < threshold_s

Function 2:
calculate_ttc_score(ttc_s, threshold_s)
returns 0.0 if TTC is too low, otherwise 1.0

Function 3:
get_ttc_reason(ttc_s, threshold_s)
returns "ttc_too_low" or "available"

main/notebook block:
ttc_s = 1.2
threshold_s = 2.0
print score and reason
```

In [ ]:
#include <iostream>
#include <string>

{
  // TODO: implement this lambda.
  auto is_ttc_too_low = [](double ttc_s, double threshold_s) -> bool {
    return false;  // replace this
  };

  // TODO: implement this lambda.
  auto calculate_ttc_score = [&](double ttc_s, double threshold_s) -> double {
    return 1.0;  // replace this
  };

  // TODO: implement this lambda.
  auto get_ttc_reason = [&](double ttc_s, double threshold_s) -> std::string {
    return "available";  // replace this
  };

  const double threshold_s = 2.0;
  double ttc_s = 1.2;

  double score = calculate_ttc_score(ttc_s, threshold_s);
  std::string reason = get_ttc_reason(ttc_s, threshold_s);

  std::cout << "score = " << score << std::endl;
  std::cout << "reason = " << reason << std::endl;
}

### Solution

In [ ]:
#include <iostream>
#include <string>

{
  auto is_ttc_too_low = [](double ttc_s, double threshold_s) -> bool {
    return ttc_s < threshold_s;
  };

  auto calculate_ttc_score = [&](double ttc_s, double threshold_s) -> double {
    if (is_ttc_too_low(ttc_s, threshold_s)) {
      return 0.0;
    }
    return 1.0;
  };

  auto get_ttc_reason = [&](double ttc_s, double threshold_s) -> std::string {
    if (is_ttc_too_low(ttc_s, threshold_s)) {
      return "ttc_too_low";
    }
    return "available";
  };

  const double threshold_s = 2.0;
  double ttc_s = 1.2;

  double score = calculate_ttc_score(ttc_s, threshold_s);
  std::string reason = get_ttc_reason(ttc_s, threshold_s);

  std::cout << "score = " << score << std::endl;
  std::cout << "reason = " << reason << std::endl;
}

Next lecture: **`struct`** — this is the big one. We’ll combine `score`, `available`, and `reason` into one object, which is exactly how real metric result code is organized.